In [3]:
import pandas as pd
import re

def get_result_metric(file_path):
    # 결과를 저장할 리스트
    best_la_auc_values = []
    best_la_aupc_values = []
    best_f1_max_values = []
    best_acc_values = []
    # 파일을 읽고 각 라인을 처리
    with open(file_path, 'r') as file:
        for line in file:
            # 'Best LA AUC' 값을 찾는다
            match = re.search(r'Best LA AUC: ([0-9.]+)', line)
            if match:
                # 찾은 값을 리스트에 추가한다
                best_la_auc_values.append(float(match.group(1)))

            match = re.search(r'Best LA AUPC: ([0-9.]+)', line)
            if match:
                # 찾은 값을 리스트에 추가한다
                best_la_aupc_values.append(float(match.group(1)))

            match = re.search(r'Best LA F1-max: ([0-9.]+)', line)
            if match:
                # 찾은 값을 리스트에 추가한다
                best_f1_max_values.append(float(match.group(1)))
            
            match = re.search(r'Best LA Accuracy: ([0-9.]+)', line)
            if match:
                # 찾은 값을 리스트에 추가한다
                best_acc_values.append(float(match.group(1)))
    
    result = {
        "ROC-AUC" : best_la_auc_values,
         "AUPC":best_la_aupc_values,
         "F1-max":best_f1_max_values,
         "Accuracy": best_acc_values if len(best_acc_values) != 0 else [0 for _ in range(len(best_f1_max_values))]
    }

    # 리스트를 사용하여 pandas DataFrame을 생성
    df = pd.DataFrame(result)
    df['ROC-AUC'] = df['ROC-AUC'].apply(lambda x: 1 - x if x < 0.5 else x)
    df['AUPC'] = df['AUPC'].apply(lambda x: 1 - x if x < 0.5 else x)
    df['F1-max'] = df['F1-max'].apply(lambda x: 1 - x if x < 0.5 else x)
    return df

def get_result(file_path, data_name):
    # 결과를 저장할 리스트
    best_la_auc_values = []

    # 파일을 읽고 각 라인을 처리
    with open(file_path, 'r') as file:
        for line in file:
            # 'Best LA AUC' 값을 찾는다
            match = re.search(r'Best LA AUC: ([0-9.]+)', line)
            if match:
                # 찾은 값을 리스트에 추가한다
                best_la_auc_values.append(float(match.group(1)))
    df = pd.DataFrame(best_la_auc_values, columns=[data_name])
    return df

import os

def gather_result(data_name):
    templates_path = ["_1_f_f_result", "_1_t_f_result", "_1_f_t_result", "_1_t_t_result",
                "_5_f_f_result", "_5_t_f_result", "_5_f_t_result", "_5_t_t_result"]
    templates_name = ["_1", "_1_mask", "_1_text", "_1_mask_text",
                      "_5", "_5_mask", "_5_text", "_5_mask_text"]
    df_list = []
    for template_name, template_path in zip(templates_name, templates_path):
        path = data_name + template_path
        name = data_name + template_name

        if not os.path.exists(path):
            continue
        df = get_result(path, name)

        df_list.append(df)

    final_df = pd.concat((df_list), axis=1)
    
    return final_df

In [3]:
def get_review_result(class_name, dir_name="review_metric"):
    one_shot = get_result_metric(file_path=f"/home/seungeon/Workspace/vlm/{dir_name}/{class_name}_1_t_t_result")
    five_shot = get_result_metric(file_path=f"/home/seungeon/Workspace/vlm/{dir_name}/{class_name}_5_t_t_result")
    df = pd.DataFrame({1: one_shot.mean(), "std(1)": one_shot.std(),"count(1)": one_shot.count(), 5: five_shot.mean(), "std(5)": five_shot.std(),"count(5)": five_shot.count()})
    return df

In [6]:
get_review_result("breakfast").transpose()

,ROC-AUC,AUPC,F1-max
1,0.846298,0.838687,0.848174
std(1),0.044472,0.047815,0.022209
count(1),50.000000,50.000000,50.000000
5,0.900164,0.889085,0.884000
std(5),0.026926,0.030401,0.021810
count(5),50.000000,50.000000,50.000000


In [7]:
get_review_result("screw_bag")

,1,std(1),count(1),5,std(5),count(5)
ROC-AUC,0.565478,0.036640,50,0.596419,0.033000,50
AUPC,0.544111,0.035446,50,0.568698,0.035149,50
F1-max,0.651543,0.007184,50,0.659134,0.010015,50


In [8]:
get_review_result("pushpins")

,1,std(1),count(1),5,std(5),count(5)
ROC-AUC,0.575759,0.042319,50,0.605006,0.034534,50
AUPC,0.663156,0.041542,50,0.686662,0.035172,50
F1-max,0.767069,0.005553,50,0.770965,0.006577,50


In [9]:
get_review_result("red_splicing")

,1,std(1),count(1),5,std(5),count(5)
ROC-AUC,0.777767,0.033340,50,0.823965,0.029973,50
AUPC,0.732062,0.051276,50,0.772918,0.048061,50
F1-max,0.776071,0.025980,50,0.811338,0.017496,50


In [10]:
get_review_result("blue_splicing")

,1,std(1),count(1),5,std(5),count(5)
ROC-AUC,0.733303,0.046110,50,0.775636,0.036620,50
AUPC,0.740004,0.065482,50,0.772766,0.053827,50
F1-max,0.800049,0.015349,50,0.823337,0.014033,50


In [11]:
get_review_result("yellow_splicing")

,1,std(1),count(1),5,std(5),count(5)
ROC-AUC,0.751410,0.032397,50,0.799503,0.020592,50
AUPC,0.769264,0.038231,50,0.814456,0.027757,50
F1-max,0.790868,0.022500,50,0.816915,0.022113,50


In [12]:
get_review_result("banana_juice")

,1,std(1),count(1),5,std(5),count(5)
ROC-AUC,0.873030,0.037042,50,0.903434,0.022652,50
AUPC,0.766144,0.078027,50,0.815773,0.050324,50
F1-max,0.766959,0.037067,50,0.798785,0.033568,50


In [13]:
get_review_result("orange_juice")

,1,std(1),count(1),5,std(5),count(5)
ROC-AUC,0.805977,0.041129,50,0.860354,0.032672,50
AUPC,0.744534,0.076440,50,0.813768,0.051805,50
F1-max,0.758361,0.028635,50,0.806181,0.031008,50


In [14]:
get_review_result("cherry_juice")

,1,std(1),count(1),5,std(5),count(5)
ROC-AUC,0.850828,0.042366,50,0.918823,0.026658,50
AUPC,0.810325,0.059995,50,0.886746,0.036547,50
F1-max,0.782787,0.037968,50,0.854163,0.034611,50


In [5]:
#Various metric Result
print(f"Reuslt of my approach")
one_shot = get_result_metric(file_path="/home/seungeon/Workspace/vlm/review_metric/breakfast_1_t_t_result")
five_shot = get_result_metric(file_path="/home/seungeon/Workspace/vlm/review_metric/breakfast_5_t_t_result")
df = pd.DataFrame({1: one_shot.mean(), "std(1)": one_shot.std(),"count(1)": one_shot.count(), 5: five_shot.mean(), "std(5)": five_shot.std(),"count(5)": five_shot.count()})

Reuslt of my approach


,1,std(1),count(1),5,std(5),count(5)
ROC-AUC,0.846298,0.044472,50,0.900164,0.026926,50
AUPC,0.838687,0.047815,50,0.889085,0.030401,50
F1-max,0.848174,0.022209,50,0.884000,0.021810,50


In [2]:
#Various metric Result
print(f"Reuslt of my approach")
one_shot = get_result_metric(file_path="/home/seungeon/Workspace/vlm/review_metric/breakfast_1_t_t_result")
five_shot = get_result_metric(file_path="/home/seungeon/Workspace/vlm/review_metric/breakfast_5_t_t_result")
pd.DataFrame({1: one_shot.mean(), 5: five_shot.mean()})

Reuslt of my approach


,1,5
ROC-AUC,0.869785,0.890940
AUPC,0.865797,0.889200
F1-max,0.854353,0.876944


In [12]:
one_shot = get_result_metric(file_path="/home/seungeon/Workspace/vlm/review_metric/breakfast_1_t_t_result")
one_shot['ROC-AUC'] = one_shot['ROC-AUC'].apply(lambda x: 1 - x if x < 0.5 else x)
one_shot

,ROC-AUC,AUPC,F1-max
0,0.869478,0.877090,0.860870
1,0.828490,0.839132,0.836820
2,0.836404,0.831697,0.837004
3,0.857193,0.846701,0.845455
4,0.877392,0.875032,0.846154
5,0.876801,0.867810,0.862385
6,0.884952,0.879029,0.858537
7,0.886605,0.876080,0.863248
8,0.881999,0.877784,0.852321
9,0.898535,0.887610,0.880734


In [10]:
#Various Zero-shot Detection Hyperparameters
kernel_space = [200, 250, 300]
t_space = [8, 10, 12]
result_df = []

for k in kernel_space:
    for t in t_space:
        for shot in [1, 5]:
            shot_path = f"/home/seungeon/Workspace/vlm/review_record/shot_{shot}_k_{k}_t_{t}.txt"
            result = get_result(file_path=shot_path, data_name=f"shot_{shot}_k_{k}_t_{t}")

            avg_score = result.mean().to_list()[0]
            avg_std = result.std().to_list()[0]
            result_df.append([shot, k, t, avg_score, avg_std])
            
result_df = pd.DataFrame(result_df, columns=["shot", "kernel_size", "threshold", "ROC-AUC", "std"], index=None)
result_df[result_df["shot"] == 1].sort_values(["shot", "kernel_size", "threshold"])

,shot,kernel_size,threshold,ROC-AUC,std
0,1,200,8,0.878759,0.020836
2,1,200,10,0.865686,0.032402
4,1,200,12,0.816678,0.056491
6,1,250,8,0.845441,0.047092
8,1,250,10,0.846575,0.035821
10,1,250,12,0.833268,0.042464
12,1,300,8,0.843692,0.019179
14,1,300,10,0.806343,0.042692
16,1,300,12,0.818627,0.033458


In [2]:
def remove_outliers_iqr(df):
    for column in df.columns:
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        # 조건을 만족하는 데이터만 반환
        df = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    return df
def visualize(class_name, outlier=False):
    if not outlier:
        one_shot = get_result_metric(file_path=f"/home/seungeon/Workspace/vlm/additional_data/{class_name}_1_t_t_result")
        five_shot = get_result_metric(file_path=f"/home/seungeon/Workspace/vlm/additional_data/{class_name}_5_t_t_result")
    else:
        one_shot = remove_outliers_iqr(get_result_metric(file_path=f"/home/seungeon/Workspace/vlm/additional_data/{class_name}_1_t_t_result"))
        five_shot = remove_outliers_iqr(get_result_metric(file_path=f"/home/seungeon/Workspace/vlm/additional_data/{class_name}_5_t_t_result"))
    df = pd.DataFrame({"1-shot(mean)": one_shot.mean(), "1-shot(std)": one_shot.std(), "1-shot(size)": one_shot.count(), 
                       "5-shot(mean)": five_shot.mean(), "5-shot(std)": five_shot.std(), "5-shot(size)": five_shot.count()})
    return df.transpose()

In [43]:
visualize("cable")

,ROC-AUC,AUPC,F1-max
1-shot(mean),0.733372,0.768698,0.784728
1-shot(std),0.050212,0.056590,0.019903
1-shot(size),50.000000,50.000000,50.000000
5-shot(mean),0.781364,0.794545,0.810905
5-shot(std),0.053725,0.059535,0.025343
5-shot(size),50.000000,50.000000,50.000000


In [42]:
visualize("capsule")

,ROC-AUC,AUPC,F1-max
1-shot(mean),0.717787,0.727425,0.768289
1-shot(std),0.054292,0.074481,0.044509
1-shot(size),50.000000,50.000000,50.000000
5-shot(mean),0.792648,0.832633,0.787873
5-shot(std),0.065969,0.052025,0.054431
5-shot(size),50.000000,50.000000,50.000000


In [41]:
visualize("transistor")

,ROC-AUC,AUPC,F1-max
1-shot(mean),0.885267,0.948019,0.933863
1-shot(std),0.051399,0.031883,0.012501
1-shot(size),50.000000,50.000000,50.000000
5-shot(mean),0.935000,0.974693,0.945791
5-shot(std),0.018876,0.010023,0.004215
5-shot(size),12.000000,12.000000,12.000000


In [17]:
visualize("pcb1")

,ROC-AUC,AUPC,F1-max
1-shot(mean),0.850872,0.860986,0.833614
1-shot(std),0.101215,0.122480,0.072174
1-shot(size),50.000000,50.000000,50.000000
5-shot(mean),0.853342,0.873696,0.821384
5-shot(std),0.084647,0.097914,0.066389
5-shot(size),50.000000,50.000000,50.000000


In [18]:
visualize("pcb2")

,ROC-AUC,AUPC,F1-max
1-shot(mean),0.648943,0.640694,0.702974
1-shot(std),0.043321,0.049661,0.013643
1-shot(size),50.000000,50.000000,50.000000
5-shot(mean),0.706538,0.713628,0.715595
5-shot(std),0.043960,0.048177,0.020089
5-shot(size),50.000000,50.000000,50.000000


In [19]:
visualize("pcb3")

,ROC-AUC,AUPC,F1-max
1-shot(mean),0.706067,0.678941,0.727770
1-shot(std),0.059737,0.058797,0.029426
1-shot(size),50.000000,50.000000,50.000000
5-shot(mean),0.747452,0.717664,0.747473
5-shot(std),0.032963,0.037141,0.022082
5-shot(size),50.000000,50.000000,50.000000


In [20]:

visualize("pcb4")

,ROC-AUC,AUPC,F1-max
1-shot(mean),0.790955,0.800712,0.759076
1-shot(std),0.083900,0.093157,0.044684
1-shot(size),50.000000,50.000000,50.000000
5-shot(mean),0.852085,0.872285,0.791960
5-shot(std),0.043633,0.047426,0.036262
5-shot(size),50.000000,50.000000,50.000000


In [6]:
def _get_review_result(class_name, dir_name="review_metric"):
    one_shot = get_result_metric(file_path=f"/home/seungeon/Workspace/vlm/{dir_name}/{class_name}_1_t_t_result")
    five_shot = get_result_metric(file_path=f"/home/seungeon/Workspace/vlm/{dir_name}/{class_name}_5_t_t_result")
    #df = pd.DataFrame({1: one_shot.mean(), "std(1)": one_shot.std(),"count(1)": one_shot.count(), 5: five_shot.mean(), "std(5)": five_shot.std(),"count(5)": five_shot.count()})
    return one_shot.mean()['Accuracy'], one_shot.std()['Accuracy'], five_shot.mean()['Accuracy'], five_shot.std()['Accuracy']

class_names = ["breakfast", "screw_bag", "pushpins"]
juice = [ "banana_juice", "orange_juice", "cherry_juice"]
connectors = ["red_splicing", "blue_splicing", "yellow_splicing"]

for c_name in class_names:
    one_acc, one_acc_std, five_Acc,five_Acc_std = _get_review_result(class_name=c_name, dir_name="accuracy")
    print("\n",c_name)
    print(one_acc, one_acc_std)
    print(five_Acc, five_Acc_std)

mean_1, mean_5 =0, 0
std_1, std_5 = 0, 0
for c_name in juice:
    one_acc, one_acc_std, five_Acc,five_Acc_std = _get_review_result(class_name=c_name, dir_name="accuracy")
    mean_1 += one_acc
    mean_5 += five_Acc
    std_1 += one_acc_std
    std_5 += five_Acc_std
print("\nJuice bottle")
print(mean_1/3, std_1/3)
print(mean_5/3, std_5/3)

mean_1, mean_5 =0, 0
std_1, std_5 = 0, 0
for c_name in connectors:
    one_acc, one_acc_std, five_Acc,five_Acc_std = _get_review_result(class_name=c_name, dir_name="accuracy")
    mean_1 += one_acc
    mean_5 += five_Acc
    std_1 += one_acc_std
    std_5 += five_Acc_std

print("\nSplicing connector")
print(mean_1/3, std_1/3)
print(mean_5/3, std_5/3)



 breakfast
0.8021621621621622 0.03188731071081649
0.8529729729729729 0.01768154295545529

 screw_bag
0.5173745173745175 0.04086102410910564
0.5583011583011583 0.03271617013793803

 pushpins
0.6393013100436681 0.01295405849274349
0.6445414847161572 0.018162999426275462

Juice bottle
0.8096234450769556 0.047555699931624486
0.8440716845878136 0.035133899125251646

Splicing connector
0.7389324227258252 0.032946226259704316
0.7541790546325763 0.022753730061889372


In [20]:
print(get_result_metric(file_path="/home/seungeon/Workspace/vlm/hybrid/pcb1_1.txt").mean()['ROC-AUC'])
get_result_metric(file_path="/home/seungeon/Workspace/vlm/hybrid/pcb1_1.txt").std()['ROC-AUC']

0.9880599999999999


0.021003047397937263

In [17]:
print(get_result_metric(file_path="/home/seungeon/Workspace/vlm/hybrid/pcb2_1.txt").mean()['ROC-AUC'])
get_result_metric(file_path="/home/seungeon/Workspace/vlm/hybrid/pcb2_1.txt").std()['ROC-AUC']

0.82057


0.01143872807614556

In [19]:
print(get_result_metric(file_path="/home/seungeon/Workspace/vlm/hybrid/pcb3_1_save.txt").mean()['ROC-AUC'])
get_result_metric(file_path="/home/seungeon/Workspace/vlm/hybrid/pcb3_1_save.txt").std()['ROC-AUC']

0.8056599999999999


0.010687983907173517

In [18]:
print(get_result_metric(file_path="/home/seungeon/Workspace/vlm/hybrid/pcb4_1.txt").mean()['ROC-AUC'])
get_result_metric(file_path="/home/seungeon/Workspace/vlm/hybrid/pcb4_1.txt").std()['ROC-AUC']

0.9457599999999999


0.008237293244749738

In [6]:
print(get_result_metric(file_path="/home/seungeon/Workspace/vlm/hybrid/cable_all_1.txt").mean()['ROC-AUC'])
get_result_metric(file_path="/home/seungeon/Workspace/vlm/hybrid/cable_all_1.txt").std()['ROC-AUC']

0.8236038230884557


0.005248212880412418

In [7]:
print(get_result_metric(file_path="/home/seungeon/Workspace/vlm/hybrid/capsule_all_1.txt").mean()['ROC-AUC'])
get_result_metric(file_path="/home/seungeon/Workspace/vlm/hybrid/capsule_all_1.txt").std()['ROC-AUC']

0.8974870362983646


0.023154126218203158

In [8]:
print(get_result_metric(file_path="/home/seungeon/Workspace/vlm/hybrid/transistor_all_1.txt").mean()['ROC-AUC'])
get_result_metric(file_path="/home/seungeon/Workspace/vlm/hybrid/transistor_all_1.txt").std()['ROC-AUC']

0.92425


0.016571080062700885